# Test Overview

1. Read the data files from github
2. Run algorithm 
3. Report results

In [3]:
import sys
print(sys.path)

['d:\\python_workspace\\Imputation', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.1008.0_x64__qbz5n2kfra8p0\\python311.zip', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.1008.0_x64__qbz5n2kfra8p0\\DLLs', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.1008.0_x64__qbz5n2kfra8p0\\Lib', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.1008.0_x64__qbz5n2kfra8p0', '', 'C:\\Users\\ch2pa\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages', 'C:\\Users\\ch2pa\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\win32', 'C:\\Users\\ch2pa\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\win32\\lib', 'C:\\Users\\ch2pa\\AppData\\Local\\Packages\\P

In [4]:
import sklearn.neighbors._base
import sys
sys.modules['sklearn.neighbors.base'] = sklearn.neighbors._base

In [5]:
from missingpy import MissForest

In [6]:
import pandas as pd
import numpy as np
n_items = 50
n_users = 50
k = 10 
ten = [str(j).zfill(2) for j in list(range(1,k+1))]
ten

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']

In [7]:
labs = ['test'+i for i in ten]
labs.insert(0,'orig')
url = "https://raw.githubusercontent.com/park61/imputation/main/data/"
url_dict = {}
url_dict["orig"] = url+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
for i in ten:
    url_dict["test{0}".format(i)] = url+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'

In [8]:
print(labs)
for lab in labs:
  print(url_dict[lab])

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_original_matrix.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data01.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data02.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data03.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data04.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data05.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data06.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data07.csv
https://raw.githubusercontent.com/park61/imputation/main/data/50-by-50_10_fold_test_data08.csv
https://raw.githubusercontent.com/park61

In [9]:
df_dict = {}
for lab in labs:
  df = pd.read_csv(url_dict[lab])
  df_dict[lab] = df.set_index('item_id')

In [10]:
# compute sparcity of the original data
#df_dict["orig"].head()
#pivot_df = df_dict["orig"].pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
n_row = df_dict["orig"].shape[0]
n_col = df_dict["orig"].shape[1]
Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
sparsity = 1 - num_Obs / (n_row * n_col)
print(sparsity)

0.18400000000000005


In [11]:
import numpy as np
col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
#print(col_max)

In [12]:
def normal1(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]/maxs[i]
  return df_out, maxs

In [13]:
def unnormal1(df, maxs):
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]*maxs[i]
  for j in range(n):
    for i in range(m):
      df_out.iloc[i,j] = round(min(maxs[j],max(1, df_out.iloc[i,j])),0)
  return df_out

In [14]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out, maxs, probs

In [15]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [16]:
#@title Default title text
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

acc_list = []
rmse_list = []
mad_list = []
time_list = []

df_orig = df_dict["orig"]
df_orig.columns = range(df_orig.shape[1])

mm = df_orig.shape[0]
nn = df_orig.shape[1]

labs_test = labs[1:]

for lab in labs_test:

  print(lab)
  
df = df_dict[lab]
import numpy as np
masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
#df, maxs = normal1(masked_df)
#df, maxs = normal2(masked_df)
#df, maxs, probs = normal3(masked_df)

from missingpy import MissForest
imputer = MissForest()

start = time_lib.time()
imputed = pd.DataFrame(imputer.fit_transform(df))

# Rounding for when no normalization           
for i in range(mm):
  for j in range(nn):
    x = imputed.iloc[i,j]
    if x < 1:
      imputed.iloc[i,j] = 1
    elif x > col_max[j]:
      imputed.iloc[i,j] = col_max[j]
    else:
      imputed.iloc[i,j] = round(x,0)

#imputed = unnormal1(imputed, maxs)
#imputed = unnormal2(imputed, maxs)
#imputed = unnormal3(imputed, maxs, probs)

end = time_lib.time()
time = end - start

df_orig.index = range(mm)
imputed.index = range(mm)

df_orig.columns = range(nn)
imputed.columns = range(nn)

diff = df_orig - imputed
sq_diff = diff ** 2
abs_diff = abs(diff)

n_match = 0
for i in range(mm):
  for j in range(nn):
    if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
      n_match += 1
acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
print(acc)  
rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
print(rmse)
print(mad)
print(time)
acc_list.append(acc)
rmse_list.append(rmse)
mad_list.append(mad)
time_list.append(time)

test01
test02
test03
test04
test05
test06
test07
test08
test09
test10
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
0.47058823529411764
1.0289915108550531
0.6862745098039216
27.261261463165283


In [17]:
print(sparsity)

print("Accuracy")
for i in acc_list:
  print(i)
print("\nRMSE")
for i in rmse_list:
  print(i)
print("\nMAD")
for i in mad_list:
  print(i)
print("\nTime")
for i in time_list:
  print(i)

0.18400000000000005
Accuracy
0.47058823529411764

RMSE
1.0289915108550531

MAD
0.6862745098039216

Time
27.261261463165283


In [18]:
acc_list

[0.47058823529411764]

In [19]:
df = pd.DataFrame(data=[acc_list, rmse_list,mad_list,time_list])
df.T

,0,1,2,3
0,0.470588,1.028992,0.686275,27.261261
